In [ ]:
from pathlib import Path
import torch
import torchvision
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

print(torch.__version__)
print(torchvision.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name())

In [ ]:
voc_root = Path("../data/VOC2012").resolve()

split_dir = voc_root/ "ImageSets" / "Segmentation"

train_ids = (split_dir/ "train.txt").read_text().split()
val_ids = (split_dir/ "val.txt").read_text().split()

print("Số ảnh train:", len(train_ids))
print("Số ảnh val:", len(val_ids))
print("5 ID đầu:", train_ids[:5])

In [ ]:
image_id = train_ids[0]

image = Image.open(voc_root / "JPEGImages" / f"{image_id}.jpg").convert("RGB")
mask = Image.open(voc_root / "SegmentationClass" / f"{image_id}.png")

image_array = np.array(image)
mask_array = np.array(mask)

fig, axes = plt.subplots(1, 2, figsize=(12,5))

axes[0].imshow(image)
axes[0].set_title(f"Original image (ID: {image_id})")

axes[1].imshow(mask)
axes[1].set_title(f"Mask for ID: {image_id}")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import xml.etree.ElementTree as ET

annotation_path = voc_root / "Annotations" / f"{image_id}.xml"
annotation = ET.parse(annotation_path).getroot()

labels = sorted(
    {
        obj.findtext("name")
        for obj in annotation.findall("object")
    }
)

print("Classes:", labels)

In [ ]:
classes = [
    "aeroplane", "bicycle", "bird", "boat", "bottle",
    "bus", "car", "cat", "chair", "cow",
    "diningtable", "dog", "horse", "motorbike", "person",
    "pottedplant", "sheep", "sofa", "train", "tvmonitor",
]

class_to_index = {
    class_name: index
    for index, class_name in enumerate(classes)
}

target = np.zeros(len(classes), dtype=np.float32)

for label in labels:
    target[class_to_index[label]] = 1.0

print("Nhãn từ XML:", labels)
print("Vector nhãn:", target)
print("Kích thước:", target.shape)

In [ ]:
rows = []

for image_id in train_ids:
    xml_path = voc_root / "Annotations" / f"{image_id}.xml"
    annotation = ET.parse(xml_path).getroot()

    image_labels = {
        obj.findtext("name")
        for obj in annotation.findall("object")
    }

    if not image_labels or not image_labels.issubset(set(classes)):
        raise ValueError

    row = {"image_id": image_id}

    for class_name in classes:
        row[class_name] = int(class_name in image_labels)

    rows.append(row)

df_train_labels = pd.DataFrame(rows, columns=["image_id", *classes])

print("Kích thước bảng:", df_train_labels.shape)
df_train_labels.head()

In [ ]:
assert df_train_labels["image_id"].is_unique

assert set(df_train_labels["image_id"]).isdisjoint(val_ids)

assert df_train_labels[classes].isin([0, 1]).all().all()
assert (df_train_labels[classes].sum(axis=1) > 0).all()

class_counts = df_train_labels[classes].sum().sort_values(ascending=False)

print(class_counts)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

class_counts.plot.barh(ax=ax)
ax.set_xlabel("Số ảnh có chứa lớp")
ax.set_ylabel("Lớp đối tượng")
ax.set_title("Phân bố nhãn trong tập train VOC2012")

plt.tight_layout()
plt.show()

print("Lớp xuất hiện nhiều nhất:", class_counts.idxmax(), class_counts.max())
print("Lớp xuất hiện ít nhất:", class_counts.idxmin(), class_counts.min())

In [ ]:
image_transform = transforms.Compose(
    [
        transforms.Resize((244, 244)),
        transforms.ToTensor(),
    ]
)

class VOCClassificationDataset(Dataset):
    def __init__(self, voc_root, labels_df, classes, transform):
        self.image_dir = voc_root / "JPEGImages"
        self.image_ids = labels_df["image_id"].tolist()
        self.targets = torch.tensor(
            labels_df[classes].to_numpy(dtype="float32")
        )
        self.transform = transform

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, index):
        image_id = self.image_ids[index]
        image_path = self.image_dir / f"{image_id}.jpg"

        with Image.open(image_path) as image:
            image = image.convert("RGB")
            image = self.transform(image)

        target = self.targets[index]

        return image, target

In [ ]:
train_dataset = VOCClassificationDataset(
    voc_root=voc_root,
    labels_df=df_train_labels,
    classes=classes,
    transform=image_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size = 8,
    shuffle=True,
    num_workers=0
)

images, targets = next(iter(train_loader))

print("Số mẫu trong dataset:", len(train_dataset))
print("Batch ảnh:", images.shape, images.dtype)
print("Batch nhãn:", targets.shape, targets.dtype)

In [ ]:
rows = []

for image_id in val_ids:
    xml_path = voc_root / "Annotations" / f"{image_id}.xml"
    annotation = ET.parse(xml_path).getroot()

    image_labels = {
        obj.findtext("name")
        for obj in annotation.findall("object")
    }

    if not image_labels or not image_labels.issubset(set(classes)):
        raise ValueError

    row = {"image_id": image_id}

    for class_name in classes:
        row[class_name] = int(class_name in image_labels)

    rows.append(row)

df_val_labels = pd.DataFrame(rows, columns=["image_id", *classes])

val_dataset = VOCClassificationDataset(
    voc_root=voc_root,
    labels_df=df_val_labels,
    classes=classes,
    transform=image_transform
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
)

In [ ]:
sample_labels = [
    classes[i]
    for i in range(len(classes))
    if targets[0, i].item() == 1
]

plt.figure(figsize=(5, 5))
plt.imshow(images[0].permute(1, 2, 0))
plt.title(", ".join(sample_labels))
plt.axis("off")
plt.show()

In [ ]:
from torch import nn
from torchvision.models import resnet18, ResNet18_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

weights = ResNet18_Weights.DEFAULT
model = resnet18(weights = weights)

print("Original output layer: ", model.fc)

model.fc = nn.Linear(
    in_features=model.fc.in_features,
    out_features=len(classes)
)

model = model.to(device)

print("Lớp đầu ra mới:", model.fc)
print("Thiết bị:", device)

In [ ]:
normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225],
)

images, targets = next(iter(train_loader))

model_inputs = normalize(images).to(device)
targets = targets.to(device)

model.eval()

with torch.no_grad():
    logits = model(model_inputs)
    probabilities = torch.sigmoid(logits)

print("Đầu vào:", model_inputs.shape)
print("Điểm số đầu ra:", logits.shape)
print("Nhãn thật:", targets.shape)
print("Xác suất của ảnh đầu:", probabilities[0].cpu())


In [ ]:
criterion = nn.BCEWithLogitsLoss()

loss = criterion(logits, targets)

print("Kích thước dự đoán:", logits.shape)
print("Kích thước nhãn thật:", targets.shape)
print("Loss:", loss.item())

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

In [ ]:
model.train()

images, targets = next(iter(train_loader))

model_inputs = normalize(images).to(device)
targets = targets.to(device)

optimizer.zero_grad()

logits = model(model_inputs)
loss = criterion(logits, targets)

loss.backward()
optimizer.step()

print("Loss của batch trước lần cập nhật:", loss.item())

In [ ]:
from tqdm.auto import tqdm

model.train()

total_loss = 0.0
total_images = 0

progress = tqdm(train_loader, desc = "Train - 1 epoch")

for images, targets in progress:
    model_inputs = normalize(images).to(device)
    targets = targets.to(device)

    optimizer.zero_grad()

    logits = model(model_inputs)
    loss = criterion(logits, targets)

    loss.backward()
    optimizer.step()

    batch_size = images.size(0)
    total_loss += loss.item() * batch_size
    total_images += batch_size

    progress.set_postfix(
        loss=f"{loss.item():.4f}",
        avg_loss=f"{total_loss / total_images:.4f}",
    )

train_loss = total_loss / total_images

print(f"Đã học qua {total_images} ảnh")
print(f"Train loss trung bình: {train_loss:.4f}")

In [ ]:
def run_epoch(loader, training):
    model.train(training)  # True: train; False: eval

    total_loss = 0.0
    total_images = 0
    description = "Train" if training else "Validation"

    with torch.set_grad_enabled(training):
        for images, targets in tqdm(loader, desc=description):
            model_inputs = normalize(images).to(device)
            targets = targets.to(device)

            if training:
                optimizer.zero_grad()

            logits = model(model_inputs)
            loss = criterion(logits, targets)

            if training:
                loss.backward()
                optimizer.step()

            batch_size = images.size(0)
            total_loss += loss.item() * batch_size
            total_images += batch_size

    return total_loss / total_images

In [ ]:
from datetime import datetime
run_dir = Path("checkpoints") / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
run_dir.mkdir(parents=True)

backbone = {
    512: "resnet18",
    2048: "resnet50"
}[model.fc.in_features]

history = []
best_val_loss = float("inf")
num_epochs = 3

for epoch in range(1, num_epochs + 1):
    print(f"\nEpoch {epoch}/{num_epochs}")

    train_loss = run_epoch(train_loader, training=True)
    val_loss = run_epoch(val_loader, training=False)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss
    })

    print(f"Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save({
            "backbone": backbone,
            "epoch_in_run": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "classes": classes,
            "val_loss": val_loss,
            "history": history,
        }, run_dir / "best.pt")
        
        print("Đã lưu checkpoint tốt hơn.")

print("\nCheckpoint:", (run_dir / "best.pt").resolve())

In [ ]:
epochs = [row["epoch"] for row in history]
train_losses = [row["train_loss"] for row in history]
val_losses = [row["val_loss"] for row in history]

plt.figure(figsize=(7,4))
plt.plot(epochs, train_losses, marker = "o", label = "Train loss")
plt.plot(epochs, val_losses, marker = "o", label = "Val loss")

plt.xlabel("Epoch trong lượt chạy này")
plt.ylabel("BCE loss")

plt.xticks(epochs)
plt.legend()
plt.grid(alpha = 0.3)
plt.tight_layout()
plt.show()

In [ ]:
checkpoint = torch.load(
    run_dir / "best.pt",
    map_location="cpu",
    weights_only=True
)

model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

print("Backbone:", checkpoint["backbone"])
print("Epoch tốt nhất trong lượt chạy:", checkpoint["epoch_in_run"])
print("Val loss:", checkpoint["val_loss"])


In [ ]:
sample_index = 5
image, target = val_dataset[sample_index]

with torch.no_grad():
    model_input = normalize(image).unsqueeze(0).to(device)
    probabilities = torch.sigmoid(model(model_input))[0].cpu()

true_labels = {
    classes[i] for i in range(len(classes)) if target[i].item() == 1
}

predicted_labels = [
    classes[i]
    for i in range(len(classes))
    if probabilities[i].item() >= 0.5
]

plt.figure(figsize=(5, 5))
plt.imshow(image.permute(1, 2, 0))
plt.axis("off")
plt.show()

print("Nhãn thật:", true_labels)
print("Dự đoán:", predicted_labels)

In [ ]:
from sklearn.metrics import average_precision_score

all_targets = []
all_scores = []

model.eval()

with torch.no_grad():
    for images, targets in tqdm(val_loader, desc="Đánh giá phân loại"):
        model_inputs = normalize(images).to(device)

        logits = model(model_inputs)
        scores = torch.sigmoid(logits)

        all_scores.append(scores.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

y_true = np.concatenate(all_targets, axis=0)
y_score = np.concatenate(all_scores, axis=0)

print("Nhãn thật:", y_true.shape)
print("Điểm dự đoán:", y_score.shape)

In [ ]:
positive_counts = y_true.sum(axis=0)

ap_per_class = average_precision_score(
    y_true,
    y_score,
    average=None,
)

classification_map = ap_per_class.mean()

ap_table = pd.DataFrame({
    "class": classes,
    "AP (%)": ap_per_class * 100,
    "positive_images": positive_counts.astype(int),
}).sort_values("AP (%)", ascending=False)

print(f"Classification mAP: {classification_map * 100:.2f}%")
ap_table